# Delta Demo — Episode 10: Schema Evolution
### "Will Adding a Column Rewrite My Delta Table?"

**Prerequisite:** Delta_Demo_Day1_Load, Day2_Load, and Day3_Load must already have been run — this episode needs a table with real accumulated history, not a fresh empty one, for the "will it rewrite millions of rows" question to carry any weight.

**Learning Outcome:** By the end of this episode, viewers should be able to predict what Delta Lake will and will not change when a new column is added.

**Core Question:** Will adding a new column rewrite my Delta table?

**How this notebook is organized:** every section is tagged with the storyboard chapter it supports (see `Episode10_Storyboard.md`). Cells marked **VERIFY** don't just print output for you to eyeball — they programmatically check a specific claim and print a pass/fail result, so nothing in this episode rests on "trust me, look closely."

# ====================
# Chapter 3 — 
#Purpose:
# Test whether ALTER TABLE rewrites the existing Parquet files.
# =====================

### Step 1 — Baseline Verification
Before changing production data, let's verify our starting point — this is a habit every engineer should have before any schema change. We'll capture the table's current rows, its full commit history, and its exact file listing, so we have something concrete to compare against after the ALTER TABLE runs.

In [0]:
%sql
-- What the table looks like right now, before any schema change.
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees` ORDER BY eno;

In [0]:
%sql
-- Full transaction history so far — note the current latest version number,
-- you'll need it later in this notebook.
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees`;

In [0]:
%python
# Capture the exact schema and exact file listing BEFORE the ALTER TABLE, into
# Python variables — not just printed to screen. We'll compare against these
# programmatically after the schema change, instead of relying on eyeballing two
# separate outputs.
import glob

table_path = "/Volumes/workspace/delta_demo/demo_files/employees"

schema_before = spark.read.format("delta").load(table_path).schema
files_before = sorted(
    f.split("/")[-1] for f in glob.glob(f"{table_path}/*.parquet")
)

print("Schema BEFORE:")
print(schema_before)
print("\nParquet files BEFORE:")
for f in files_before:
    print(" -", f)

### Step 2 — Run the Schema Change
HR wants an Age column. Let's give it to them.

In [0]:
%sql
ALTER TABLE delta.`/Volumes/workspace/delta_demo/demo_files/employees` ADD COLUMNS (age INT);

### Step 3 — Query Immediately
No backfill statement has been run. Watch what the existing rows show for `age`.

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees` ORDER BY eno;

### DESCRIBE HISTORY — Confirm a New Commit Exists

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees`;

### Browse `_delta_log` — Confirm a New JSON File Appeared

In [0]:
%sh
ls -la /Volumes/workspace/delta_demo/demo_files/employees/_delta_log/*.json

### Automatically Find and Read the Newest Commit
Rather than hardcoding a version number (which will differ depending on how many commits your own table has accumulated), this cell finds the newest `_delta_log` JSON file automatically and reads it. Look for exactly one action type in the output: `metaData`. No `add`, no `remove`.

In [0]:
%python
import glob, json

log_files = sorted(glob.glob(f"{table_path}/_delta_log/*.json"))
latest_commit_path = log_files[-1]
print(f"Reading latest commit: {latest_commit_path.split('/')[-1]}\n")

action_types = []
with open(latest_commit_path) as f:
    for line in f:
        action = json.loads(line)
        action_types.append(list(action.keys())[0])
        print(json.dumps(action, indent=2))
        print("---")

print(f"\nAction types found in this commit: {action_types}")

### VERIFY — This Commit Contains ONLY a Schema Change
Programmatic check: the latest commit's actions should be exactly `commitInfo` and `metaData` — nothing else. If `add` or `remove` shows up here, the claim "this was metadata-only" would be false, and this cell will say so explicitly rather than silently passing.

In [0]:
%python
has_add_or_remove = "add" in action_types or "remove" in action_types
has_metadata = "metaData" in action_types

if has_metadata and not has_add_or_remove:
    print("✅ VERIFIED: latest commit contains a metaData action and NO add/remove actions.")
    print("   This confirms the schema change did not touch any data files.")
else:
    print("❌ NOT VERIFIED — check the commit contents above manually.")
    print(f"   Action types found: {action_types}")

### VERIFY — Data Files Are Byte-for-Byte Unchanged
Programmatic check: compare the exact file listing captured in the baseline (Chapter 3, Step 1) against the current file listing. Same file names expected — if even one file name differs, that would mean something was physically rewritten.

In [0]:
%python
files_after = sorted(
    f.split("/")[-1] for f in glob.glob(f"{table_path}/*.parquet")
)

print("Parquet files BEFORE:", files_before)
print("Parquet files AFTER: ", files_after)

if files_before == files_after:
    print("\n✅ VERIFIED: identical file names before and after — no files were")
    print("   added, removed, or rewritten by the schema change.")
else:
    print("\n❌ NOT VERIFIED — file listing changed. Investigate before recording.")
    print("   Added:", set(files_after) - set(files_before))
    print("   Removed:", set(files_before) - set(files_after))

### VERIFY — Schema Actually Changed
The other half of the claim: while data files stayed the same, the table's *logical* schema should now include `age`.

In [0]:
%python
schema_after = spark.read.format("delta").load(table_path).schema

print("Schema BEFORE:", schema_before)
print("Schema AFTER: ", schema_after)

if "age" in schema_after.fieldNames() and "age" not in schema_before.fieldNames():
    print("\n✅ VERIFIED: 'age' column is present after ALTER TABLE, absent before.")
else:
    print("\n❌ NOT VERIFIED — schema did not change as expected.")